# Plant Disease Classifier
## Step 4: Build the Model

**What you will learn in this notebook:**
- Package the model into a clean reusable function
- Move model to GPU automatically
- Verify the full model pipeline
- Save everything as `src/model.py`

---


---
## Cell 1: Import Libraries

In [1]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {device}')
if torch.cuda.is_available():
    print(f'GPU name        : {torch.cuda.get_device_name(0)}')

PyTorch version : 2.5.1+cu121
Device          : cuda
GPU name        : NVIDIA GeForce RTX 4050 Laptop GPU


---
## Cell 2: What is `device` and Why Does it Matter?

In PyTorch, everything (model + data) must be on the **same device**.

```
device = 'cuda'   means use GPU  (fast)
device = 'cpu'    means use CPU  (slow)

Rule: model and data must always be on the same device
      model.to(device) moves the model
      images.to(device) moves the data
```

Since you have an RTX 4050, your device will be `cuda` and training will be fast.

In [2]:
# Demonstrate moving tensors between devices
x = torch.randn(3, 224, 224)   # created on CPU by default
print(f'Tensor device before : {x.device}')

x = x.to(device)               # move to GPU
print(f'Tensor device after  : {x.device}')

print()
print('Same must be done for our model and all image batches during training.')

Tensor device before : cpu
Tensor device after  : cuda:0

Same must be done for our model and all image batches during training.


---
## Cell 3: Build the Model Function

Instead of writing the same code every time, we package everything into one clean function called `build_model()`.

It does 4 things:
1. Load MobileNetV3 with pretrained weights
2. Freeze the feature layers
3. Replace the classifier for binary output
4. Move model to GPU

In [3]:
def build_model(num_classes=2, device='cpu'):
    """
    Builds and returns a MobileNetV3 Small model for binary classification.

    Args:
        num_classes : number of output classes (2 for healthy/diseased)
        device      : 'cuda' for GPU, 'cpu' for CPU

    Returns:
        model       : ready to train MobileNetV3 model
    """

    # Step 1: Load pretrained MobileNetV3 Small
    model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)

    # Step 2: Freeze all feature layers
    # These layers already know how to detect edges, textures, shapes
    for param in model.features.parameters():
        param.requires_grad = False

    # Step 3: Replace the classifier
    # Original: Linear(1024 -> 1000)  for ImageNet
    # New     : Linear(1024 -> 2)     for our task
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)

    # Step 4: Move model to device (GPU or CPU)
    model = model.to(device)

    return model


print('build_model() function defined successfully.')

build_model() function defined successfully.


---
## Cell 4: Create the Model and Verify

In [4]:
# Build the model
model = build_model(num_classes=2, device=device)

# Count parameters
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print('Model built successfully.')
print()
print(f'Device          : {next(model.parameters()).device}')
print(f'Total params    : {total:,}')
print(f'Frozen params   : {frozen:,}   (features - not updated during training)')
print(f'Trainable params: {trainable:,}   (classifier - updated during training)')
print(f'Trainable %     : {trainable/total*100:.1f}%')

Model built successfully.

Device          : cuda:0
Total params    : 1,519,906
Frozen params   : 927,008   (features - not updated during training)
Trainable params: 592,898   (classifier - updated during training)
Trainable %     : 39.0%


---
## Cell 5: Verify the Classifier

Let's confirm the last layer outputs 2 values (not 1000).

In [5]:
print('Final classifier structure:')
print('----------------------------')
print(model.classifier)
print()

last_layer = model.classifier[-1]
print(f'Last layer input  : {last_layer.in_features}')
print(f'Last layer output : {last_layer.out_features}  <-- should be 2')

Final classifier structure:
----------------------------
Sequential(
  (0): Linear(in_features=576, out_features=1024, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1024, out_features=2, bias=True)
)

Last layer input  : 1024
Last layer output : 2  <-- should be 2


---
## Cell 6: Test Forward Pass with Real Batch Size

Let's simulate exactly what happens during training.
We feed a batch of 32 fake images and check the output shape.

In [6]:
# Simulate a real training batch
# Shape: [batch_size=32, channels=3, height=224, width=224]
fake_batch = torch.randn(32, 3, 224, 224).to(device)   # move to same device as model

model.eval()
with torch.no_grad():
    output = model(fake_batch)

print('Forward pass with batch of 32 images:')
print('--------------------------------------')
print(f'Input shape  : {fake_batch.shape}')
print(f'Output shape : {output.shape}   <-- [32 images, 2 class scores each]')
print()
print('Output for first 3 images (raw scores):')
print(output[:3])
print()

# Convert to probabilities
probs = torch.softmax(output, dim=1)
print('Probabilities for first 3 images:')
for i in range(3):
    print(f'  Image {i+1}: Healthy={probs[i][0].item()*100:.1f}%  Diseased={probs[i][1].item()*100:.1f}%')

Forward pass with batch of 32 images:
--------------------------------------
Input shape  : torch.Size([32, 3, 224, 224])
Output shape : torch.Size([32, 2])   <-- [32 images, 2 class scores each]

Output for first 3 images (raw scores):
tensor([[-0.1383, -0.1147],
        [-0.0927, -0.1885],
        [-0.1232, -0.1628]], device='cuda:0')

Probabilities for first 3 images:
  Image 1: Healthy=49.4%  Diseased=50.6%
  Image 2: Healthy=52.4%  Diseased=47.6%
  Image 3: Healthy=51.0%  Diseased=49.0%


---
## Cell 7: Print Trainable Layer Names

Let's see exactly which layers will be updated during training.

In [7]:
print('Trainable layers (will be updated during training):')
print('----------------------------------------------------')
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f'  {name:50s}  shape: {list(param.shape)}')

print()
print('Frozen layers (will NOT be updated):')
print('--------------------------------------')
frozen_count = 0
for name, param in model.named_parameters():
    if not param.requires_grad:
        frozen_count += 1
print(f'  {frozen_count} frozen parameter tensors in features layers')

Trainable layers (will be updated during training):
----------------------------------------------------
  classifier.0.weight                                 shape: [1024, 576]
  classifier.0.bias                                   shape: [1024]
  classifier.3.weight                                 shape: [2, 1024]
  classifier.3.bias                                   shape: [2]

Frozen layers (will NOT be updated):
--------------------------------------
  138 frozen parameter tensors in features layers


---
## Cell 8: Save as src/model.py

We save the `build_model()` function to `src/model.py` so we can import it in the training notebook.

In [8]:
import os
os.makedirs('../src', exist_ok=True)

model_code = '''
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights


def build_model(num_classes=2, device='cpu'):
    """
    Builds MobileNetV3 Small for binary plant disease classification.

    Args:
        num_classes : number of output classes (default 2)
        device      : 'cuda' or 'cpu'

    Returns:
        model : MobileNetV3 ready for training
    """
    model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)

    for param in model.features.parameters():
        param.requires_grad = False

    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)

    model = model.to(device)
    return model


if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model  = build_model(num_classes=2, device=device)
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model built on {device}')
    print(f'Total params    : {total:,}')
    print(f'Trainable params: {trainable:,}')
'''

with open('../src/model.py', 'w') as f:
    f.write(model_code)

print('src/model.py saved successfully.')
print()
print('Step 4 Complete!')
print('Next: Step 5 - Train the Model')

src/model.py saved successfully.

Step 4 Complete!
Next: Step 5 - Train the Model


In [11]:
import shutil, os

shutil.move('./src/dataset.py', '../src/dataset.py')
print('dataset.py moved successfully.')

# Verify
print()
print('Inside project root src/:')
for f in os.listdir('../src'):
    print(f'   {f}')




dataset.py moved successfully.

Inside project root src/:
   dataset.py
   model.py


---
## Step 4 Summary

| Concept | What it is | Why it matters |
|---------|-----------|----------------|
| `device` | cuda or cpu | Model and data must be on same device |
| `.to(device)` | moves tensor/model to device | Required before training |
| `build_model()` | reusable function | Clean way to create model anywhere |
| `requires_grad=False` | freezes a layer | Preserved pretrained knowledge |
| Forward pass | image in, scores out | Core operation of neural networks |
| `softmax` | converts scores to probabilities | Makes output interpretable |

---
